In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("src")

import torch
import gc
import pandas as pd
from tqdm import tqdm

import _prompt
import _mapping
import _util
from _intervention import get_label_probability

In [3]:
model_type = "GPT-OSS" # GPT-OSS or R1

if model_type == "GPT-OSS":
    model, tokenizer = _util.load_OSS()
elif model_type == "R1":
    model, tokenizer = _util.load_R1()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [4]:
prompt_type = "h_pre_result" # empty or pre_result or pre_sum
if prompt_type:
    prompt_type = "_" + prompt_type

# Load the divided prompts dataset
if 'h' in prompt_type:
    prompts = pd.read_csv(f"data/{model_type}/h_prompts{prompt_type[2:]}.csv")
else:
    prompts = pd.read_csv(f"data/{model_type}/prompts{prompt_type}.csv")
prompts["base_sum"] = prompts["base_sum"].astype('Int64')
prompts["source_sum"] = prompts["source_sum"].astype('Int64')
print(f"loaded {len(prompts)} divided prompts")

loaded 256 divided prompts


In [7]:
intervention_loc = [8,10,12] # restatement or reasoning or restatement_and_reasoning

if type(intervention_loc) == str:
    if 'h' in prompt_type:
        if model_type == "GPT-OSS":
            intervention_ids_dict = _mapping.intervene_ids_stepwise_3_digit_h
        elif model_type == "R1":
            intervention_ids_dict = _mapping.intervene_ids_R1_3_digit_h
    else:
        if model_type == "GPT-OSS":
            intervention_ids_dict = _mapping.intervene_ids_stepwise_3_digit
        elif model_type == "R1":
            intervention_ids_dict = _mapping.intervene_ids_R1_3_digit

    if intervention_loc == "restatement":
        intervention_ids = intervention_ids_dict["restatement"]
    elif intervention_loc == "reasoning":
        intervention_ids = intervention_ids_dict["reasoning"]
    elif intervention_loc == "restatement_and_reasoning":
        intervention_ids = intervention_ids_dict["restatement"] + intervention_ids_dict["reasoning"]
elif type(intervention_loc) == list:
    intervention_ids = intervention_loc

print(intervention_ids)

[8, 10, 12]


In [8]:
# Get header of divided prompts dataset
header = list(prompts.columns) + ['intervention_ids', 'intervention_id', 'intervention_prompt', 'generated_text']

filepath = _util.create_csv_file(f"experiments/token_intervention/output/{model_type}/h_result_unfaithful_source", f"{prompt_type[1:]}_restatement.csv", header, overwrite=False)

batch_size = 24

for i in tqdm(range(0, len(prompts), batch_size)):
    torch.cuda.empty_cache()
    gc.collect()
    batch_rows = prompts.iloc[i:i+batch_size]

    # factual_labels = batch_rows['base_sum']
    # counterfactual_labels = batch_rows['source_sum']
    
    # Prepare batch of intervention prompts
    intervention_prompts = [_prompt.get_intervened_prompt(intervention_ids, row['base_prompt'], row['source_prompt']) for _, row in batch_rows.iterrows()]
    
    # Tokenize all prompts in the batch
    tokens = tokenizer(intervention_prompts, add_special_tokens=False, return_tensors="pt", padding=True, padding_side="left").to(model.device)

    # Generate for the entire batch
    generations = model.generate(
        tokens.input_ids,
        attention_mask=tokens.attention_mask,
        max_new_tokens=2,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    
    # base_labels = tokenizer([str(base_sum) for base_sum in factual_labels], add_special_tokens=False, return_tensors="pt", padding=True, padding_side="right")["input_ids"].to(model.device)
    # base_labels_mask = base_labels != tokenizer.pad_token_id
    # base_labels_probs = get_label_probability(model, tokens, base_labels, base_labels_mask)

    # source_labels = tokenizer([str(source_sum) for source_sum in counterfactual_labels], add_special_tokens=False, return_tensors="pt", padding=True, padding_side="right")["input_ids"].to(model.device)
    # source_labels_mask = source_labels != tokenizer.pad_token_id
    # source_labels_probs = get_label_probability(model, tokens, source_labels, source_labels_mask)
    
    # Process each generated text in the batch
    for j, (_, row) in enumerate(batch_rows.iterrows()):
        intervention_prompt = intervention_prompts[j]
        generated_text = tokenizer.decode(generations[j]).replace(intervention_prompt, "").replace(tokenizer.pad_token[-1], "")
        _util.write_to_csv(filepath, row.to_list() + [intervention_ids, ', '.join(str(id) for id in intervention_ids), intervention_prompt, generated_text]) #, base_labels_probs[j].item(), source_labels_probs[j].item()])


  0%|                                                                             | 0/11 [00:00<?, ?it/s]

100%|████████████████████████████████████████████████████████████████████| 11/11 [00:59<00:00,  5.40s/it]
